In [2]:
with open('./train/neg.txt', 'r', encoding='utf-8') as f:
    neg_data = f.read()
with open('./train/pos.txt', 'r', encoding='utf-8') as f:
    pos_data = f.read()

neg_datalist = neg_data.split('\n')
pos_datalist = pos_data.split('\n')
len(neg_datalist), len(pos_datalist)

(422, 417)

In [3]:
import numpy as np

# 在读取到数据后，我们将将数据存到一个列表中，并构建标签列表，用 1 表示正面的评论，用 0 表示负面的评论。
dataset = np.array(pos_datalist + neg_datalist)
labels = np.array([1] * len(pos_datalist) + [0] * len(neg_datalist))
len(dataset)  # 共 839 条数据

839

In [4]:
# 利用 NumPy 库使样本数据随机排列。
np.random.seed(10)
mix_index = np.random.choice(839, 839)
dataset = dataset[mix_index]
labels = labels[mix_index]
len(dataset), len(labels)

(839, 839)

In [5]:
# 取 700 条数据作为训练集，取 139 条数据作为验证集。
TRAINSET_SIZE = 700
EVALSET_SIZE = 139
 
train_samples = dataset[:TRAINSET_SIZE]  # 700 条数据
train_labels = labels[:TRAINSET_SIZE]
eval_samples = dataset[TRAINSET_SIZE:TRAINSET_SIZE+EVALSET_SIZE]  # 139 条数据
eval_labels = labels[TRAINSET_SIZE:TRAINSET_SIZE+EVALSET_SIZE]
 
len(train_samples), len(eval_samples)

(700, 139)

In [6]:
# 构建函数 get_dummies ，作用是把标签转换成 one-hot 的表示形式，例如将 1 表示成 [0, 1]，0 表示成 [1, 0] 的形式。
def get_dummies(l, size=2):
    res = list()
    for i in l:
        tmp = [0] * size
        tmp[i] = 1
        res.append(tmp)
    return res

In [14]:
!pip install transformers

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 10.4/10.4 MB 1.7 MB/s eta 0:00:00
   ---------------------------------------- 484.3/484.3 kB 2.5 MB/s eta 0:00:00
   ---------------------------------------- 308.9/308.9 kB 2.4 MB/s eta 0:00:00
   ---------------------------------------- 2.4/2.4 MB 2.2 MB/s eta 0:00:00
   ---------------------------------------- 194.4/194.4 kB 2.0 MB/s eta 0:00:00


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 23.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
# Bert_Base_Chinese的下载地址：https://huggingface.co/bert-base-chinese
# 自动下载模型与分词器方法
from transformers import BertModel, BertTokenizer

# 联网下载模型与分词器使用，地址存放在C:\Users\admin\.cache\huggingface\hub\models--bert-base-chinese

model_name = 'bert-base-chinese'
tokenizer = BertTokenizer.from_pretrained(model_name)

sents = ["白日依山尽", "黄河入海流"]
out = tokenizer.encode(
    # 传入的两个句子
    text=sents[0],
    text_pair=sents[1],
    # 长度大于设置是否截断
    truncation=True,
    # 一律补齐，如果长度不够
    padding='max_length',
    add_special_tokens=True,
    max_length=30,
    return_tensors=None,
)
print(out)
print(tokenizer.decode(out))

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

C:\Users\Jessie\AppData\Roaming\Python\Python39\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Jessie\.cache\huggingface\hub\models--bert-base-chinese. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/269k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

[101, 4635, 3189, 898, 2255, 2226, 102, 7942, 3777, 1057, 3862, 3837, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[CLS] 白 日 依 山 尽 [SEP] 黄 河 入 海 流 [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


In [17]:
# 使用 PyTorch 提供的 DataLoader() 构建训练集数据集表示，使用 TensorDataset() 构建训练集数据迭代器。

import torch
from transformers import BertTokenizer
from torch.utils.data import DataLoader, TensorDataset

model_name = 'bert-base-chinese'  # 指定需下载的预训练模型参数
tokenizer = BertTokenizer.from_pretrained(model_name) 
tokenized_text = [tokenizer.tokenize(i) for i in train_samples]
input_ids = [tokenizer.convert_tokens_to_ids(i) for i in tokenized_text]
input_labels = get_dummies(train_labels)  # 使用 get_dummies 函数转换标签
 
for j in range(len(input_ids)):
    # 将样本数据填充至长度为 512
    i = input_ids[j]
    if len(i) != 512:
        input_ids[j].extend([0]*(512 - len(i)))
        
# 构建数据集和数据迭代器，设定 batch_size 大小为 4
train_set = TensorDataset(torch.LongTensor(input_ids),
                          torch.FloatTensor(input_labels))
train_loader = DataLoader(dataset=train_set,
                          batch_size=4,
                          shuffle=True)
train_loader

In [18]:
# 与构建训练集数据迭代器类似，构建验证集的数据迭代器。
tokenized_text = [tokenizer.tokenize(i) for i in eval_samples]
input_ids = [tokenizer.convert_tokens_to_ids(i) for i in tokenized_text]
input_labels = eval_labels
 
for j in range(len(input_ids)):
    i = input_ids[j]
    if len(i) != 512:
        input_ids[j].extend([0]*(512 - len(i)))

eval_set = TensorDataset(torch.LongTensor(input_ids),
                         torch.FloatTensor(input_labels))
eval_loader = DataLoader(dataset=eval_set,
                         batch_size=1,
                         shuffle=True)
eval_loader

In [19]:
# 检查是否机器有 GPU，如果有就在 GPU 运行，否则就在 CPU 运行
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [21]:
# 构建一个用于分类的类，加入 BERT 模型，在 BERT 模型下加入一个 Dropout 层用于防止过拟合，和一个 Linear 全连接层。
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertModel
 

class fn_cls(nn.Module):
    def __init__(self):
        super(fn_cls, self).__init__()
        self.model = BertModel.from_pretrained(model_name, cache_dir="./")
        self.model.to(device)
        self.dropout = nn.Dropout(0.1)
        self.l1 = nn.Linear(768, 2)
 
    def forward(self, x, attention_mask=None):
        outputs = self.model(x, attention_mask=attention_mask)
        x = outputs[1]  # 取池化后的结果 batch * 768
        x = x.view(-1, 768)
        x = self.dropout(x)
        x = self.l1(x)
        return x

In [22]:
# 定义损失函数，建立优化器。
from torch import optim
 
cls = fn_cls()
cls.to(device)
cls.train()
 
criterion = nn.BCELoss()
sigmoid = nn.Sigmoid()
optimizer = optim.Adam(cls.parameters(), lr=1e-5)

config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

C:\Users\Jessie\AppData\Roaming\Python\Python39\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Jessie\Documents\thesis\models--bert-base-chinese. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP dow

model.safetensors:   0%|          | 0.00/412M [00:00<?, ?B/s]

In [23]:
# 构建预测函数，用于计算预测结果。
def predict(logits):
    res = torch.argmax(logits, 1)
    return res

In [24]:
'''
构建训练函数并开始训练。这里需要说一下，因为 GPU 内存的限制，训练集的 batch_size 设为了 4，
这样的 batch_size 过小，使得梯度下降方向不准，引起震荡，难以收敛。
所以，在训练时使用了梯度积累的方法，即计算 8 个小批次的梯度的平均值来更新模型，从而达到了 32 个小批次的效果。
'''
from torch.autograd import Variable
import time
 
pre = time.time()
 
accumulation_steps = 8
epoch = 3
 
for i in range(epoch):
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = Variable(data).to(device), Variable(target.view(-1, 2)).to(device)
 
        mask = []
        for sample in data:
            mask.append([1 if i != 0 else 0 for i in sample])
        mask = torch.Tensor(mask).to(device)
        
        output = cls(data, attention_mask=mask)
        pred = predict(output)
 
        loss = criterion(sigmoid(output).view(-1, 2), target)
 
        # 梯度积累
        loss = loss/accumulation_steps
        loss.backward()
 
        if((batch_idx+1) % accumulation_steps) == 0:
            # 每 8 次更新一下网络中的参数
            optimizer.step()
            optimizer.zero_grad()
 
        if ((batch_idx+1) % accumulation_steps) == 1:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss:{:.6f}'.format(
                i+1, batch_idx, len(train_loader), 100. *
                batch_idx/len(train_loader), loss.item()
            ))
        if batch_idx == len(train_loader)-1:
            # 在每个 Epoch 的最后输出一下结果
            print('labels:', target)
            print('pred:', pred)
            
print('训练时间：', time.time()-pre)

Train Epoch: 1 [0/175 (0%)]	Loss:0.068244
Train Epoch: 1 [8/175 (5%)]	Loss:0.073425
Train Epoch: 1 [16/175 (9%)]	Loss:0.104289
Train Epoch: 1 [24/175 (14%)]	Loss:0.074011
Train Epoch: 1 [32/175 (18%)]	Loss:0.103391
Train Epoch: 1 [40/175 (23%)]	Loss:0.079940
Train Epoch: 1 [48/175 (27%)]	Loss:0.068073
Train Epoch: 1 [56/175 (32%)]	Loss:0.076129
Train Epoch: 1 [64/175 (37%)]	Loss:0.083155
Train Epoch: 1 [72/175 (41%)]	Loss:0.050787
Train Epoch: 1 [80/175 (46%)]	Loss:0.074722
Train Epoch: 1 [88/175 (50%)]	Loss:0.038651
Train Epoch: 1 [96/175 (55%)]	Loss:0.054644
Train Epoch: 1 [104/175 (59%)]	Loss:0.046536
Train Epoch: 1 [112/175 (64%)]	Loss:0.051925
Train Epoch: 1 [120/175 (69%)]	Loss:0.038980
Train Epoch: 1 [128/175 (73%)]	Loss:0.054005
Train Epoch: 1 [136/175 (78%)]	Loss:0.055558
Train Epoch: 1 [144/175 (82%)]	Loss:0.030182
Train Epoch: 1 [152/175 (87%)]	Loss:0.027966
Train Epoch: 1 [160/175 (91%)]	Loss:0.028868


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# 训练结束后，可以使用验证集观察模型的训练效果。
from tqdm import tqdm_notebook as tqdm
 
cls.eval()
 
correct = 0
total = 0
 
for batch_idx, (data, target) in enumerate(tqdm(eval_loader)):
    data = data.to(device)
    target = target.long().to(device)
 
    mask = []
    for sample in data:
        mask.append([1 if i != 0 else 0 for i in sample])
    mask = torch.Tensor(mask).to(device)
 
    output = cls(data, attention_mask=mask)
    pred = predict(output)
 
    correct += (pred == target).sum().item()
    total += len(data)

# 准确率应该达到百分之 90 以上
print('正确分类的样本数：{}，样本总数：{}，准确率：{:.2f}%'.format(
    correct, total, 100.*correct/total))

In [ ]:
# 训练结束后，还可以随意输入一些数据，直接观察模型的预测结果。
test_samples = ['东西很好，好评！', '东西不好，差评！']
cls.eval()
tokenized_text = [tokenizer.tokenize(i) for i in test_samples]
input_ids = [tokenizer.convert_tokens_to_ids(i) for i in tokenized_text]
input_ids = torch.LongTensor(input_ids).cuda()
 
mask = torch.ones_like(input_ids).to(device)
 
output = cls(input_ids, attention_mask=mask)
pred = predict(output)
pred